# Project settings

In [1]:
import os
from pathlib import Path

project_root = Path(os.getcwd()).parent.resolve()
Path(project_root).resolve()
os.chdir(project_root)

In [ ]:
import pandas as pd
from src import config


2026-03-17 23:48:31.234 | INFO     | ieee_cis_fraud_detection.config:<module>:11 - PROJ_ROOT path is: /home/laenra/projects/ieee-cis-fraud-detection


# Read data

List transaction data

In [3]:
train_transaction_files= [
    os.path.join(
        config.INTERIM_DATA_DIR,
        'train_transaction_chunks',
        fn
    )
    for fn in os.listdir(os.path.join(config.INTERIM_DATA_DIR, 'train_transaction_chunks'))
]

identity_files= [
    os.path.join(
        config.INTERIM_DATA_DIR,
        'train_identity_chunks',
        fn
    )
    for fn in os.listdir(os.path.join(config.INTERIM_DATA_DIR, 'train_identity_chunks'))
]

print(f"Total number of train_transaction files: {len(train_transaction_files)}")
print(f"Total number of identity files: {len(identity_files)}")

Total number of train_transaction files: 60
Total number of identity files: 15


# Preprocess and write as tfrecord

In [ ]:
import re
import numpy as np
import tensorflow as tf
from pathlib import Path

tfrecord_dir = Path(config.INTERIM_DATA_DIR) / "train_tfrecord_chunks"
tfrecord_dir.mkdir(parents=True, exist_ok=True)

def _bytes_feature(value):
    if isinstance(value, str):
        value = value.encode("utf-8")
    elif isinstance(value, bytes):
        pass
    else:
        value = str(value).encode("utf-8")
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=[float(value)]))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))

def row_to_example(row):
    features = {}
    for col, val in row.items():
        if pd.isna(val):
            continue
        if isinstance(val, (np.integer, int, np.int64)):
            features[col] = _int64_feature(val)
        elif isinstance(val, (np.floating, float, np.float64)):
            features[col] = _float_feature(val)
        else:
            features[col] = _bytes_feature(val)
    return tf.train.Example(features=tf.train.Features(feature=features))

identity_map = {
    int(re.search(r"_chunk_(\d+)\.parquet$", f).group(1)): f
    for f in identity_files
}

for tx_file in train_transaction_files:
    match = re.search(r"_chunk_(\d+)\.parquet$", tx_file)
    if not match:
        continue
    chunk_id = int(match.group(1))

    df_tx = pd.read_parquet(tx_file)
    ident_file = identity_map.get(chunk_id)
    if ident_file:
        df_id = pd.read_parquet(ident_file)
        df = df_tx.merge(df_id, on="TransactionID", how="left")
    else:
        df = df_tx

    # basic preprocessing
    df = df.replace({np.nan: None})
    for col in df.select_dtypes(include=["category", "object"]).columns:
        df[col] = df[col].fillna("").astype(str)
    for col in df.select_dtypes(include=["int64", "float64"]).columns:
        df[col] = df[col].fillna(0)

    out_file = tfrecord_dir / f"train_chunk_{chunk_id}.tfrecord"
    with tf.io.TFRecordWriter(str(out_file)) as writer:
        for _, row in df.iterrows():
            example = row_to_example(row)
            writer.write(example.SerializeToString())

print("Finished writing TFRecord files to", tfrecord_dir)